# EXP-DENSENET121-TRANSFER-5SEEDS-021 — real holdout (4)

**Evaluation only.** Load the same five Validation-locked DenseNet121
checkpoints from EXP-018 and score `RL_Cashew_holdout (4).zip` once. There is
no training and no internal Final Test in this notebook. Output goes to a
new EXP-020 directory; EXP-018 and EXP-019 remain untouched.

Holdout (4) has 126 images across four true classes: 46 anthracnose,
19 healthy, 36 leaf miner and 25 red rust. `not_cashew_leaf` remains a possible
prediction, but has zero true support. Macro metrics use four supported true
classes. The notebook checks the ZIP SHA-256 and exact byte overlap with all
of Split 9 before inference.

Compared with Holdout (2), this ZIP removes `healthy/real_healthy_028.jpg`
and changes the bytes of eight other healthy images. Scores from the two
versions must retain their separate dataset hashes.

Run all cells in order on Colab. The configuration finds the project shortcut
in the current account's My Drive. If this runtime still has the old account
mounted, set `FORCE_REMOUNT_DRIVE=True`. Use the path overrides if the shared
folder is elsewhere or read-only.


In [1]:
# 01. Imports and fixed evaluation protocol
import base64
import csv
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
import zlib
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from tensorflow import keras
from tensorflow.keras import layers

SOURCE_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-018"
EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-020"
SEEDS = [42, 123, 2026, 3407, 7777]
CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
HOLDOUT_CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust"]
SPLIT_ZIP = "CashewData_Split (9).zip"
HOLDOUT_ZIP = "RL_Cashew_holdout (3).zip"
SPLIT_SHA256 = "1eb8c46921de19ab53ef8d50e05d8785f94dcb45a5511dcf32e2e7e01516c6f4"
HOLDOUT_SHA256 = "134072def34a6d766068b7c2de68085e3a6876d8544450d51b9428d0dfa6f6c9"
EXPECTED_HOLDOUT_COUNTS = {"anthracnose": 46, "healthy": 19,
                           "leaf_miner": 36, "red_rust": 25}
IMG_SIZE, BATCH_SIZE = 224, 16
NUM_CLASSES = len(CLASSES)
HEAD_DROPOUT = 0.40
WEIGHT_DECAY = 1e-4
FORCE_REMOUNT_DRIVE = False
DRIVE_PROJECT_DIR_OVERRIDE = None
SOURCE_RESULTS_ROOT_OVERRIDE = None
DRIVE_RESULTS_ROOT_OVERRIDE = None
DATASET_ZIP_OVERRIDE = None
HOLDOUT_ZIP_OVERRIDE = None

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("TensorFlow", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


TensorFlow 2.21.0 GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# 02. Mount the current account and locate source/output directories
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive", force_remount=FORCE_REMOUNT_DRIVE)
    drive_root = Path("/content/drive/MyDrive")
    if DRIVE_PROJECT_DIR_OVERRIDE is not None:
        project_dir = Path(DRIVE_PROJECT_DIR_OVERRIDE).expanduser()
    else:
        direct = drive_root / "Cashew_Leaf_model_result"
        candidates = [direct] if (direct / "Dataset" / SPLIT_ZIP).is_file() else []
        if not candidates:
            candidates = [p for p in drive_root.iterdir() if p.is_dir() and
                          (p / "Dataset" / SPLIT_ZIP).is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Found {len(candidates)} project shortcuts. Set DRIVE_PROJECT_DIR_OVERRIDE.")
        project_dir = candidates[0]
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = (Path(SOURCE_RESULTS_ROOT_OVERRIDE).expanduser() if
                   SOURCE_RESULTS_ROOT_OVERRIDE else default_results)
    results_root = (Path(DRIVE_RESULTS_ROOT_OVERRIDE).expanduser() if
                    DRIVE_RESULTS_ROOT_OVERRIDE else default_results)
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else project_dir / "Dataset" / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else project_dir / "Dataset" / HOLDOUT_ZIP
    cache_root = Path("/content/dataset_cache/exp020")
elif os.name == "nt":
    project_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Resource")
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = Path(SOURCE_RESULTS_ROOT_OVERRIDE) if SOURCE_RESULTS_ROOT_OVERRIDE else default_results
    results_root = Path(DRIVE_RESULTS_ROOT_OVERRIDE) if DRIVE_RESULTS_ROOT_OVERRIDE else default_results
    dataset_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset")
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else dataset_dir / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else dataset_dir / HOLDOUT_ZIP
    cache_root = project_dir / "dataset_cache" / "exp020"
else:
    raise RuntimeError("EXP-020 supports Colab or Windows; set explicit paths for another runtime.")

source_dir = source_root / SOURCE_ID
out_dir = results_root / EXPERIMENT_ID
aggregate_dir = out_dir / "aggregate"
figures_dir = out_dir / "figures"
for directory in (out_dir, aggregate_dir, figures_dir):
    directory.mkdir(parents=True, exist_ok=True)
probe = out_dir / ".write_probe"
try:
    probe.write_text("ok", encoding="utf-8")
finally:
    probe.unlink(missing_ok=True)
if not source_dir.is_dir():
    raise FileNotFoundError(f"EXP-018 result directory not found: {source_dir}")
for path in (split_archive, holdout_archive):
    if not path.is_file():
        raise FileNotFoundError(path)
print("EXP-018:", source_dir)
print("EXP-020:", out_dir)
print("Split ZIP:", split_archive)
print("Holdout ZIP:", holdout_archive)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXP-018: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-018
EXP-020: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-020
Split ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/CashewData_Split (9).zip
Holdout ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/RL_Cashew_holdout (3).zip


## Verify the EXP-018 lock and revised holdout archive

Any archive, manifest or checkpoint mismatch stops before holdout inference.


In [3]:
# 03. Verify recipe, selection lock, five checkpoints and archive identities
config_path = source_dir / "experiment_config.json"
lock_path = source_dir / "validation_selection_lock.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
lock = json.loads(lock_path.read_text(encoding="utf-8"))
if config.get("experiment_id") != SOURCE_ID or lock.get("experiment_id") != SOURCE_ID:
    raise ValueError("Source experiment identity mismatch")
if config.get("dataset", {}).get("archive_sha256", "").lower() != SPLIT_SHA256:
    raise ValueError("Source recipe uses a different dataset ZIP")
if lock.get("dataset_sha256", "").lower() != SPLIT_SHA256:
    raise ValueError("Validation lock uses a different dataset ZIP")
if lock.get("config_sha256", "").lower() != sha256_file(config_path):
    raise ValueError("Source config changed after Validation lock")
if config.get("dataset", {}).get("class_names") != CLASSES or lock.get("class_names") != CLASSES:
    raise ValueError("Class order changed")
if config.get("seeds") != SEEDS or lock.get("seeds") != SEEDS:
    raise ValueError("Five-seed protocol changed")
if lock.get("dataset_manifest_sha256") != config["dataset"]["manifest_sha256"]:
    raise ValueError("Source manifest fingerprint mismatch")
if sha256_file(split_archive) != SPLIT_SHA256 or sha256_file(holdout_archive) != HOLDOUT_SHA256:
    raise ValueError("Split 9 or Holdout 3 archive SHA-256 mismatch")
checkpoint_paths = {seed: source_dir / f"seed_{seed}" / "best.weights.h5" for seed in SEEDS}
checkpoint_hashes = {}
for seed, path in checkpoint_paths.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256_file(path)
    if actual.lower() != lock["selected_checkpoint_sha256"][str(seed)].lower():
        raise ValueError(f"Checkpoint changed: seed {seed}")
    checkpoint_hashes[str(seed)] = actual
deployment_seed = int(lock["deployment_seed"])
if deployment_seed not in SEEDS:
    raise ValueError("Invalid deployment seed in source lock")
validation_csv = source_dir / "aggregate" / "validation_results_5seeds.csv"
if sha256_file(validation_csv) != lock["validation_sha256"]:
    raise ValueError("Validation results changed after lock")
validation_rows = pd.read_csv(validation_csv)
if validation_rows["seed"].tolist() != SEEDS:
    raise ValueError("Source Validation results are incomplete")
print("Source lock PASS; deployment seed:", deployment_seed)


Source lock PASS; deployment seed: 123


In [4]:
# 04. Safe extraction and exact manifest verification
def extract_frozen_zip(archive_path, target_parent, root_name, archive_sha):
    target = target_parent / root_name
    marker = target_parent / ".archive_sha256"
    if target_parent.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != archive_sha:
            raise RuntimeError(f"Unverified extraction cache: {target_parent}")
    else:
        staging = target_parent.with_name(target_parent.name + "_staging")
        if staging.exists():
            raise RuntimeError(f"Partial extraction requires review: {staging}")
        staging.mkdir(parents=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                name = member.filename.replace("\\", "/")
                parts = Path(name).parts
                resolved = (staging / name).resolve()
                if not parts or parts[0] != root_name or staging.resolve() not in resolved.parents:
                    raise ValueError(f"Unsafe or unexpected ZIP member: {member.filename}")
            archive.extractall(staging)
        (staging / ".archive_sha256").write_text(archive_sha, encoding="utf-8")
        staging.rename(target_parent)
    if not target.is_dir():
        raise FileNotFoundError(target)
    return target

split_root = extract_frozen_zip(split_archive, cache_root / "split09", "CashewData_Split", SPLIT_SHA256)
holdout_root = extract_frozen_zip(holdout_archive, cache_root / "holdout03", "RL_Cashew_holdout", HOLDOUT_SHA256)
source_manifest_path = source_dir / "dataset_manifest.csv"
source_manifest = pd.read_csv(source_manifest_path, dtype={"sha256": str, "relative_path": str})
needed_columns = {"relative_path", "label", "sha256", "split", "class"}
if not needed_columns.issubset(source_manifest.columns) or len(source_manifest) != 7045:
    raise ValueError("EXP-018 source manifest is missing or incomplete")
manifest_records = [{"relative_path": str(row.relative_path), "label": int(row.label),
                     "sha256": str(row.sha256)}
                    for row in source_manifest.itertuples(index=False)]
manifest_digest = hashlib.sha256(json.dumps(
    manifest_records, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
if manifest_digest != lock["dataset_manifest_sha256"]:
    raise ValueError("EXP-018 source manifest content changed")
reference_hashes = set()
for row in source_manifest.itertuples(index=False):
    relative = Path(row.relative_path)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe source manifest path: {row.relative_path}")
    image = split_root / relative
    if not image.is_file() or sha256_file(image) != row.sha256:
        raise ValueError(f"Split 9 extracted image changed: {row.relative_path}")
    reference_hashes.add(row.sha256)
val_manifest = source_manifest[source_manifest["split"] == "val"].head(2).copy()
val_manifest["image_path"] = val_manifest["relative_path"].map(lambda x: str(split_root / x))

holdout_rows = []
for name in HOLDOUT_CLASSES:
    folder = holdout_root / name
    files = sorted(p for p in folder.iterdir() if p.is_file())
    if len(files) != EXPECTED_HOLDOUT_COUNTS[name]:
        raise ValueError(f"Holdout class count changed: {name}")
    for image in files:
        with Image.open(image) as opened:
            opened.verify()
        digest = sha256_file(image)
        if digest in reference_hashes:
            raise ValueError(f"Holdout image overlaps Split 9: {image}")
        holdout_rows.append({"relative_path": image.relative_to(holdout_root).as_posix(),
                             "image_path": str(image), "class": name,
                             "label": CLASSES.index(name), "sha256": digest})
holdout_manifest = pd.DataFrame(holdout_rows)
if len(holdout_manifest) != 126 or holdout_manifest["sha256"].duplicated().any():
    raise ValueError("Holdout 3 contains missing or duplicate images")
holdout_manifest.to_csv(out_dir / "holdout_dataset_manifest.csv", index=False)
save_json(out_dir / "dataset_audit.json", {
    "split_zip_sha256": SPLIT_SHA256, "source_manifest_sha256": manifest_digest,
    "holdout_zip_sha256": HOLDOUT_SHA256,
    "holdout_counts": EXPECTED_HOLDOUT_COUNTS, "holdout_exact_overlap": 0,
    "limitation": "Near duplicates and shared leaf/tree/session IDs need separate review.",
})
print("Dataset audit PASS: Holdout 3", len(holdout_manifest))


Dataset audit PASS: Holdout 3 126


## Rebuild the EXP-018 model graph and load frozen weights

The model definition is copied from EXP-018. `weights=None` only avoids
redownloading ImageNet weights; the trained checkpoint is loaded immediately.


In [5]:

# ============================================================
# 04. REPRODUCIBILITY
# ============================================================

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Đồng bộ Python / NumPy / TensorFlow theo Keras.
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

    print(f"Global seed set to: {seed}")


try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic operations: ENABLED")
except Exception as e:
    print("Deterministic operations: unavailable")
    print("Reason:", e)

# 08. ImageNet backbone, frozen BN, and smoke test
import tempfile
from PIL import ImageOps

AUGMENTATION_CONFIG = {"horizontal_flip": True, "rotation_factor": 0.08,
                       "zoom_factor": 0.10, "contrast_factor": 0.10,
                       "brightness": False}

def densenet_preprocessing(inputs):
    x = layers.Rescaling(1.0 / 255.0, name="densenet_rescale")(inputs)
    return layers.Normalization(mean=[0.485, 0.456, 0.406],
                                variance=[0.229**2, 0.224**2, 0.225**2],
                                axis=-1, name="densenet_preprocess")(x)

def build_transfer_model(seed, weights="imagenet"):
    set_global_seed(seed)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="raw_rgb")
    augmented = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(0.08, seed=seed),
        layers.RandomZoom(0.10, seed=seed),
        layers.RandomContrast(0.10, seed=seed),
    ], name=f"train_only_augmentation_seed_{seed}")(inputs)
    x = densenet_preprocessing(augmented)
    backbone = keras.applications.DenseNet121(include_top=False, weights=weights,
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    backbone.trainable = False
    features = backbone(x, training=False)  # Keep BN moving statistics fixed in both stages.
    x = layers.Dropout(HEAD_DROPOUT, seed=seed)(features)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                           name="disease_output")(x)
    return keras.Model(inputs, outputs, name=f"DenseNet121_Transfer_seed_{seed}"), backbone

def set_backbone_stage(backbone, stage):
    if stage == "stage1":
        backbone.trainable = False
    elif stage == "stage2":
        backbone.trainable = True
        for layer in backbone.layers:
            layer.trainable = (layer.name.startswith("conv5_") and
                               not isinstance(layer, layers.BatchNormalization))
        if not any(layer.trainable and layer.weights for layer in backbone.layers):
            raise RuntimeError("No weighted conv5_* layer was unfrozen")
        if any(layer.trainable for layer in backbone.layers
               if isinstance(layer, layers.BatchNormalization)):
            raise RuntimeError("A backbone BatchNorm layer is trainable")
    else:
        raise ValueError(stage)
    return int(sum(np.prod(v.shape) for v in backbone.trainable_weights))

def compile_stage(model, lr):
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr,
                                                  weight_decay=WEIGHT_DECAY),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"], steps_per_execution=1, jit_compile=False)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def load_locked_model(seed):
    keras.backend.clear_session()
    gc.collect()
    model, backbone = build_transfer_model(seed, weights=None)
    selection = json.loads((source_dir / f"seed_{seed}" / "stage_selection.json").read_text(encoding="utf-8"))
    stage = selection["selected_stage"]
    if stage not in {"stage1", "stage2"}:
        raise ValueError(f"Unexpected selected stage for seed {seed}: {stage}")
    if selection.get("checkpoint_sha256", "").lower() != checkpoint_hashes[str(seed)]:
        raise ValueError(f"Stage selection/checkpoint fingerprint mismatch: seed {seed}")
    set_backbone_stage(backbone, stage)
    model.load_weights(checkpoint_paths[seed])
    return model

def make_eval_dataset(frame):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["label"].to_numpy(dtype=np.int32)
    def decode(path, label):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
        return tf.cast(image, tf.float32), label
    return tf.data.Dataset.from_tensor_slices((paths, labels)).map(
        decode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify the selected seed model against the source .keras deployment package.
source_deployment = source_dir / "model_package" / "deployment_model.keras"
if not source_deployment.is_file():
    raise FileNotFoundError(source_deployment)
selected_model = load_locked_model(deployment_seed)
reference_model = keras.models.load_model(source_deployment, compile=False)
sample = next(iter(make_eval_dataset(val_manifest)))[0]
if not np.allclose(selected_model(sample, training=False).numpy(),
                   reference_model(sample, training=False).numpy(), atol=1e-5):
    raise RuntimeError("Rebuilt checkpoint disagrees with EXP-018 deployment model")
del selected_model, reference_model
keras.backend.clear_session(); gc.collect()
print("Checkpoint/model parity PASS")


Deterministic operations: ENABLED
Global seed set to: 123
Checkpoint/model parity PASS


In [6]:
# 06. Freeze the holdout evaluation inputs before inference
evaluation_lock = {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "source_validation_results_sha256": sha256_file(validation_csv),
    "source_manifest_sha256": manifest_digest,
    "split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "selected_checkpoint_sha256": checkpoint_hashes,
    "seeds": SEEDS, "deployment_seed": deployment_seed,
    "class_order": CLASSES, "selection_source": "EXP-018 Validation only",
    "holdout_images": 126, "internal_test": "NOT_RUN",
}
evaluation_lock_path = out_dir / "evaluation_lock.json"
if evaluation_lock_path.exists():
    if json.loads(evaluation_lock_path.read_text(encoding="utf-8")) != evaluation_lock:
        raise RuntimeError("EXP-020 evaluation lock changed; do not mix datasets or checkpoints")
else:
    save_json(evaluation_lock_path, evaluation_lock)
print("Evaluation lock PASS")


Evaluation lock PASS


## External real holdout (3)

The five EXP-018 checkpoints stay fixed. All five class probabilities are
exported per image; macro scores use the four true classes in this holdout.


In [7]:
# 07. Evaluate every frozen checkpoint once and export standard atomic outputs
PROB_COLUMNS = [f"prob_{name}" for name in CLASSES]

def write_matrix(matrix, output, normalized):
    suffix = "normalized" if normalized else "raw"
    pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(
        output / f"confusion_matrix_{suffix}.csv")
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=40, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                    ha="center", va="center", fontsize=9)
    ax.set(xlabel="Predicted", ylabel="True", title=f"{output.parent.name} / {output.name}")
    fig.tight_layout(); fig.savefig(output / f"confusion_matrix_{suffix}.png", dpi=180); plt.close(fig)

def write_calibration(predictions, output):
    confidence = predictions["confidence"].to_numpy(dtype=float)
    correct = predictions["correct"].to_numpy(dtype=bool)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    rows = []
    edges = np.linspace(0, 1, 11)
    for i in range(10):
        mask = (confidence >= edges[i]) & (confidence < edges[i+1] if i < 9 else confidence <= 1)
        n = int(mask.sum())
        rows.append({"bin": i, "n": n, "lower": edges[i], "upper": edges[i+1],
                     "accuracy": float(correct[mask].mean()) if n else np.nan,
                     "mean_confidence": float(confidence[mask].mean()) if n else np.nan})
    bins = pd.DataFrame(rows)
    bins.to_csv(output / "calibration_bins.csv", index=False)
    ece = float(sum(r["n"] / len(predictions) * abs(r["accuracy"] - r["mean_confidence"])
                    for r in rows if r["n"]))
    brier = float(np.mean(np.sum((probs - np.eye(NUM_CLASSES)[truth]) ** 2, axis=1)))
    save_json(output / "calibration_metrics.json", {"ece_10_bins": ece,
              "multiclass_brier": brier, "n": len(predictions),
              "note": "Diagnostic only; no threshold or calibration fit."})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    used = bins[bins["n"] > 0]
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].plot(used["mean_confidence"], used["accuracy"], "o-")
    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Confidence", ylabel="Accuracy",
                title=f"Reliability; ECE={ece:.3f}")
    axes[1].hist(confidence[correct], bins=edges, alpha=.7, label="Correct")
    axes[1].hist(confidence[~correct], bins=edges, alpha=.7, label="Incorrect")
    axes[1].set(xlabel="Confidence", ylabel="Images", title="Confidence distribution")
    axes[1].legend(); fig.tight_layout()
    fig.savefig(output / "reliability_confidence.png", dpi=180); plt.close(fig)
    return ece, brier

def write_roc_pr(predictions, output):
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for index, name in enumerate(CLASSES):
        binary = (truth == index).astype(int)
        if binary.sum() in (0, len(binary)):
            rows.append({"class": name, "support": int(binary.sum()), "roc_auc": np.nan,
                         "average_precision": np.nan, "status": "UNDEFINED"})
            continue
        fpr, tpr, _ = roc_curve(binary, probs[:, index])
        precision, recall, _ = precision_recall_curve(binary, probs[:, index])
        auc_score = float(roc_auc_score(binary, probs[:, index]))
        ap = float(average_precision_score(binary, probs[:, index]))
        rows.append({"class": name, "support": int(binary.sum()),
                     "roc_auc": auc_score, "average_precision": ap, "status": "OK"})
        axes[0].plot(fpr, tpr, label=f"{name} {auc_score:.3f}")
        axes[1].plot(recall, precision, label=f"{name} {ap:.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC OVR")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR OVR")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(output / "roc_pr_ovr.png", dpi=180); plt.close(fig)
    pd.DataFrame(rows).to_csv(output / "roc_pr_ovr_metrics.csv", index=False)

def score_split(split_name, frame, seed, model, output):
    dataset = make_eval_dataset(frame)
    _ = model.predict(dataset.take(1), verbose=0)
    started = time.perf_counter()
    probabilities = np.asarray(model.predict(dataset, verbose=0))
    elapsed = time.perf_counter() - started
    y_true = frame["label"].to_numpy(dtype=int)
    if probabilities.shape != (len(frame), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError("Prediction matrix has wrong shape or non-finite values")
    if not np.allclose(probabilities.sum(axis=1), 1, atol=1e-4):
        raise RuntimeError("Prediction probabilities do not sum to one")
    y_pred = probabilities.argmax(axis=1)
    supported = [CLASSES.index(x) for x in HOLDOUT_CLASSES]
    report = classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                   target_names=CLASSES, output_dict=True, zero_division=0)
    report_frame = pd.DataFrame(report).T
    if split_name == "real_holdout":
        report_frame.loc["not_cashew_leaf", ["precision", "recall", "f1-score"]] = np.nan
    report_frame.to_csv(output / "classification_report.csv")
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_norm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
    write_matrix(cm, output, False)
    write_matrix(cm_norm, output, True)
    relative = frame["relative_path"].astype(str).to_numpy()
    predictions = pd.DataFrame({
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "architecture": "DenseNet121", "training_mode": "ImageNet Transfer, two-stage",
        "dataset_version": "RL_Cashew_holdout_03", "dataset_sha256": HOLDOUT_SHA256,
        "split": split_name, "seed": seed, "image_id": relative, "relative_path": relative,
        "true_label": [CLASSES[i] for i in y_true],
        "predicted_label": [CLASSES[i] for i in y_pred],
        "true_class_idx": y_true, "predicted_class_idx": y_pred,
        "correct": y_true == y_pred, "confidence": probabilities.max(axis=1),
    })
    for index, name in enumerate(CLASSES):
        predictions[f"prob_{name}"] = probabilities[:, index]
    predictions.to_csv(output / "predictions.csv", index=False)
    predictions.loc[~predictions["correct"]].to_csv(output / "error_cases.csv", index=False)
    ece, brier = write_calibration(predictions, output)
    write_roc_pr(predictions, output)
    disease = np.isin(y_true, [CLASSES.index(x) for x in ("anthracnose", "leaf_miner", "red_rust")])
    false_healthy = int(np.sum(disease & (y_pred == CLASSES.index("healthy"))))
    metrics = {
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "split": split_name, "seed": seed, "n": len(frame),
        "macro_class_count": len(supported), "supported_true_classes": [CLASSES[i] for i in supported],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=supported,
                                                    average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=supported,
                                              average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=supported,
                                      average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "loss": float(-np.log(np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-7, 1)).mean()),
        "false_healthy_count": false_healthy, "disease_images": int(disease.sum()),
        "false_healthy_rate": float(false_healthy / disease.sum()),
        "ece_10_bins": ece, "multiclass_brier": brier,
        "prediction_seconds_including_input_pipeline": elapsed,
        "inference_ms_per_image_batch16": 1000 * elapsed / len(frame),
        "checkpoint_sha256": checkpoint_hashes[str(seed)],
    }
    save_json(output / "metrics.json", metrics)
    return metrics

all_metrics = {}
for split_name, frame in (("real_holdout", holdout_manifest),):
    rows = []
    for seed in SEEDS:
        output = out_dir / f"seed_{seed}" / split_name
        marker = output / "evaluation_completed.json"
        if marker.is_file():
            saved = json.loads(marker.read_text(encoding="utf-8"))
            if saved.get("evaluation_lock_sha256") != sha256_file(evaluation_lock_path):
                raise RuntimeError(f"Completed {split_name} seed {seed} has a different lock")
            metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))
            if not (output / "predictions.csv").is_file():
                raise RuntimeError(f"Completed {split_name} seed {seed} is missing predictions")
            print("Reusing completed", split_name, seed)
        else:
            if output.exists() and any(output.iterdir()):
                raise RuntimeError(f"Partial evaluation requires inspection: {output}")
            output.mkdir(parents=True, exist_ok=True)
            model = load_locked_model(seed)
            metrics = score_split(split_name, frame, seed, model, output)
            del model
            keras.backend.clear_session(); gc.collect()
            save_json(marker, {"evaluation_lock_sha256": sha256_file(evaluation_lock_path),
                               "checkpoint_sha256": checkpoint_hashes[str(seed)],
                               "split": split_name, "seed": seed,
                               "completed_at_utc": datetime.now(timezone.utc).isoformat()})
        rows.append(metrics)
    if [row["seed"] for row in rows] != SEEDS:
        raise RuntimeError("Five-seed evaluation incomplete")
    all_metrics[split_name] = pd.DataFrame(rows)
    all_metrics[split_name].to_csv(aggregate_dir / f"{split_name}_results_5seeds.csv", index=False)
print("Holdout 3 inference complete")


Global seed set to: 42


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 123
Global seed set to: 2026


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 3407


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 7777


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Holdout 3 inference complete


In [8]:
# 08. Aggregate summaries and visual panels
for split_name in ("real_holdout",):
    results = all_metrics[split_name]
    metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1",
                    "balanced_accuracy", "loss", "false_healthy_rate",
                    "ece_10_bins", "multiclass_brier", "inference_ms_per_image_batch16"]
    summary = pd.DataFrame([{"metric": name, "mean": float(results[name].mean()),
                             "sample_sd": float(results[name].std(ddof=1)),
                             "n_seeds": len(SEEDS), "images_per_seed": int(results.iloc[0]["n"])}
                            for name in metric_names])
    summary.to_csv(aggregate_dir / f"{split_name}_mean_std.csv", index=False)
    per_class = []
    for seed in SEEDS:
        report = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                             "classification_report.csv", index_col=0)
        for name in CLASSES:
            per_class.append({"split": split_name, "seed": seed, "class": name,
                              "support": int(report.loc[name, "support"]),
                              "precision": report.loc[name, "precision"],
                              "recall": report.loc[name, "recall"],
                              "f1": report.loc[name, "f1-score"]})
    per_class = pd.DataFrame(per_class)
    per_class.to_csv(aggregate_dir / f"per_class_{split_name}_by_seed.csv", index=False)
    per_class.groupby("class")[["precision", "recall", "f1"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / f"per_class_{split_name}_mean_std.csv")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    primary = summary[summary["metric"].isin(["accuracy", "macro_recall", "macro_f1", "balanced_accuracy"])]
    ax.bar(primary["metric"], primary["mean"], yerr=primary["sample_sd"], capsize=5)
    ax.set(ylim=(0, 1.05), ylabel="Score", title=f"{split_name}: mean ± sample SD across five seeds")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_metrics_mean_seed_sd.png", dpi=180); plt.close(fig)
    heat = per_class.pivot(index="class", columns="seed", values="f1").reindex(CLASSES)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    image = ax.imshow(heat, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(SEEDS)), SEEDS)
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    ax.set(xlabel="Seed", title=f"{split_name}: per-class F1")
    for r in range(NUM_CLASSES):
        for c in range(len(SEEDS)):
            value = heat.iloc[r, c]
            ax.text(c, r, f"{value:.2f}" if np.isfinite(value) else "N/A",
                    ha="center", va="center")
    fig.colorbar(image, ax=ax, label="F1")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_per_class_f1_heatmap.png", dpi=180); plt.close(fig)
    for normalized in (False, True):
        suffix = "normalized" if normalized else "raw"
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(27, 5.5))
        for ax, seed in zip(axes, SEEDS):
            matrix = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                                 f"confusion_matrix_{suffix}.csv", index_col=0).to_numpy()
            ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
            ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=50, ha="right")
            ax.set_yticks(range(NUM_CLASSES), CLASSES)
            ax.set_title(f"seed {seed}")
            for r in range(NUM_CLASSES):
                for c in range(NUM_CLASSES):
                    ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                            ha="center", va="center", fontsize=8)
        fig.tight_layout()
        fig.savefig(figures_dir / f"{split_name}_confusion_{suffix}_5seeds.png", dpi=180)
        plt.close(fig)

print("Holdout 3 aggregate tables and figures complete")


Holdout 3 aggregate tables and figures complete


In [9]:
# 09. Report, manifest and archives; source checkpoints included in FULL ZIP
import base64
import zlib

readme = f"""# {EXPERIMENT_ID}

Holdout-only continuation of {SOURCE_ID}. No training, Final Test or holdout-based selection.
Source Validation lock: `{sha256_file(lock_path)}`. Deployment seed {deployment_seed}.
Split 9 SHA-256 `{SPLIT_SHA256}`; holdout 3 SHA-256 `{HOLDOUT_SHA256}`.
Holdout 3: 126 images, four true classes. Internal Final Test is NOT_RUN here.
The two macro metrics use different class support. `not_cashew_leaf` holdout
F1 is undefined because it has zero true support.
Relative to Holdout (2), one healthy image was removed and eight
healthy images changed bytes; compare versions with that provenance.

Per-seed outputs: `seed_*/real_holdout/` contains metrics, standard
predictions with five probabilities, errors, reports, confusion CSV/PNG,
calibration ECE/Brier and ROC/PR one-vs-rest. Aggregate summaries and panels
are in `aggregate/` and `figures/`. SD is sample SD across five seeds, not an
image-sampling confidence interval. Prediction timing is batch {BATCH_SIZE}
including data loading after one-batch warm-up, and should not be compared
across different devices as a deployment latency benchmark.

`FULL.zip` adds the five EXP-018 source checkpoints and deployment model;
`ANALYSIS.zip` contains reports, notebook and figures without model binaries.
The source EXP-018 result directory remains unmodified.
"""
(out_dir / "README.md").write_text(readme, encoding="utf-8")
save_json(out_dir / "execution_status.json", {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "training": "REUSED_EXP018", "final_test": "NOT_RUN", "real_holdout": "COMPLETE",
    "source_validation_lock_sha256": sha256_file(lock_path),
    "evaluation_lock_sha256": sha256_file(evaluation_lock_path),
})
required = ["README.md", "execution_status.json", "evaluation_lock.json", "dataset_audit.json",
            "holdout_dataset_manifest.csv",
            "aggregate/real_holdout_results_5seeds.csv", "aggregate/real_holdout_mean_std.csv"]
for seed in SEEDS:
    for split_name in ("real_holdout",):
        required += [f"seed_{seed}/{split_name}/{name}" for name in
                     ("metrics.json", "predictions.csv", "classification_report.csv",
                      "confusion_matrix_raw.csv", "confusion_matrix_raw.png",
                      "confusion_matrix_normalized.csv", "confusion_matrix_normalized.png",
                      "calibration_metrics.json", "roc_pr_ovr_metrics.csv",
                      "evaluation_completed.json")]
missing = [name for name in required if not (out_dir / name).is_file() or
           (out_dir / name).stat().st_size == 0]
save_json(out_dir / "audit_required_artifacts.json", {
    "experiment_id": EXPERIMENT_ID, "required_count": len(required),
    "missing": missing, "status": "PASS" if not missing else "FAIL"})
if missing:
    raise RuntimeError(f"Required evaluation artifacts missing: {missing}")

NOTEBOOK_SOURCE_ZLIB_B64 = "eNrNfe1y4ziS4KvgPBE7VI1MW/4ql3q1ES7b1eXtKtthu3pu2lawKImyOSWJWpKyrXJo4x7hIu7+3GPc7/13cS+y9ySXHwAIgKQs18fG9u50WySYSCQSmYnMROJprR+NRtla+/qJ/gry+TRaa6+Nw/TLIHmYrDXXxlEeDsI8XGs/LZprWTJL+9Dieu1P4vi/nq8fHZ9eHp8eX7W2WutXFwenl++OL9Z3L4+Pjy7XN7c2xf/7b/9TpFE4EnfJaJDMcuFtN24QLP3r1avj+3A0C/M4mYhkMpr7r16JD0k4EPldJLJwHIlhfB+J38NRPKBW66Ok/yUaiKNokkWnUQ79EqT+XdT/Mk3iSZ6JYZqMCbnN1r4IJwOR9ZM0Ep8vPgSHYXYXPQQGMv7XePoZ+u5Hvri6i6BdnBHESSLyNIwn8eSWgMBvgB6lExjMuxj/fRVlOTwDXOMMXudRL0m++OJslk8B9m0SZSJPRMjQogdGCWgyiNOonyfp/BcLS/77DZBrDN2K2SRPZjCsgV/Q632Bt7gLM9Ha2hPxOLyFnsJ+mmQwdpgfwHsWif4ozLIoa4udPQCf36Vhf5JkUZMAQTd3MCv53bwptvfEKAqHYhxPopQw2doFJAYinWW5Lz7DyII+0w3bfZYIQpdiCl3GvVFEMKfwTdzHSWqKHiCJCH6N0oTRyWbTaZICvI+IqQCuSuN+JmZZxDjL99AtNuc55RHQvGj6CprpjBjkj5Nzcfn+YH1rd4/wjh7Dfi568zwSyX2UjsKpeIjzOxGORgQwGYrL6SjOxRvRi4bIE/FkCFOOk18Q+TAZT0McP32rSb7VaPJMY69AAughE58lFTeQxwP5I9jc2vf/Pr39TOAQsf5dOMFJQqQRvUwAKlF8e5eLBJ6lajLkZPriEjlWcjJ+lD8kBAxGlQGBMzGGuQEs8pD4L4qBgBFgHcLQcbFmEdH/DmAVA7uYTZAUgpY8Mm6SDqBvWHqHySjsMZ37yWQY385SXpPDeDJgtKdp8nfgWpHdwSz1ZzlB5M5Ff5YCEXPgwX4CbPvnTHyci6MUVq4vToZMtRRexLCeszwGFJA38EsgrvqKAI7xr2jQFDiAz+/OLg6Pg4vjj2efTq+Co4uT3487V8Adn33xCfiG0AphjnCy03gABIuHLDnucAIJ4hC6gEECBtEoix5ohScpyqTBOkkcaNVdNG3x108GEXwbPUYwUqBDQBiutSez0ciViAktdxShXVs8brZg9GNk6oy4cxg/AlNFhcADkuZJP2HejKml6MHU7e2YT/rZvfnztm/+wjkexT3z0d+zZGL+TjLz13QU5sD5Y/NZCtgl1pPsDoZt4ZXNLTA4leZvEKLDeGQ/UngREwO9Ipp/+Vb9bhKor8kkKtrinMLHquk5/CyYWD4ch/l0lOTQzJ/O8S8BHDUd5WabyWw8nePzydQiAQwXHmL7gTUmUClJOhwlD/guHxYInZ98UMic4AJt8n/OplnRJvsCwjGd+EqwyfYecDcspv48ICXUFCHwKnwbgLTsx7iW5QuCJJb90wtHIUiqQeCCJDkZD+M+sVWQRtjzCgBxqc8IBaBmGj82xbClYBbowR8gNALo8n4VLN1xCfm9+pX0g3DWN38S5EZBSWMeJBG/AMmyqgY+vVHNRuEcxGPBKpdnn1B+nByJjrhZe9Zaae3f8HfQ8vji5OMxyJxVv93alN/Sb/jmemerCep5uym2Nrf2QMnubL5uitfwT5eVzIeDy8tjanmzZqjnm7Um9CfVAf9ApRuQcubfINkCVM38y9HON2sM//3Zh6OzT1fBD+mHQV6efzi5ClD7IUnYkjoCMRiwTvXekC0l6aD6V81rbS9FNwIOyhx1OX7Qinr7/Z29N1utQdR6E/Z2t6Ph/mB3M9rcHey/3t8dvtkZ9Hs7u+Hubqs16A+3t6Kt6HW02dpt7fX3hjsOHiZknIutQTTc3gn3Bq/39jb39nuv+/Bkb39zfzfaDvf2X+8N9nd3dnZ2Nwe7rd6bna39weZgGO4N9/pvDC45vDo+CjSpUUshpZ8cSqMJZlG7LVpvnl1K9oS0wUyzJ6UNZtqCgJx8/DW4PPnjuCneHlwdvqe/AY2trR1gwT1qcvrpo8EKo2jiyV+87t4fHxyBfj07h3HA+01/Z5Oe//X45Nf3oHmPDw/+Bs9b0TqrpgrFDK/fhaBh6T09Cc4vzv4ZSBQcnVwEZ78fX1ycHGGzUyXs5fq8OL789OHqMrg4O7uqbMfQnm92cHVweUwsV/neYMnK9/QvYAu0HoBXAlRoHuqiRlvP1SC+RaO/oxSvz229hm5BRiOqLP7UT6ZAbJi33s1aA/VKloPpMW5bsw8KWfRwY4NGWQx7DG8UjnuDsC1b+2itePviH/8RpAmYoQALoLVLHMTY+bMpalePIBaIgbE4SyeqzV30yH95DWfwoKACtCEI/6ZAcyUy+iKDq2OM0Hrjo+U8yf3xF9jkePwjI5utCdZ5nOVB8oV+Op89pDDsII8ecw/79gegujOP+m4CUQYApgMCFQT/LI2CMOvHcYfYrSH+Aqvi5maCMgsM+WQA+7XOzdosH66DPDcGN01h+wYzcUXa4x1oD/wkH/pBIK3qIMAV9uv5p0y+YVvYHyHe07t5Bhp2FAyi+7gfZR61hB4aP8+A3ILdEn5XZWSTSQkzjBY/f7TBgPT2Mo5YF+bpvJg+UqC3SXI7imB4YPVrgwzNdWoWPfajaS5N1+M0TVKD/7GVuWjA4OZnvAWmF05zn4x6oNcG0BN0d77BfQGNgfH7UYCbKWjRqZAqDRtUkCZJrrjPBbjxcX7EgIuvAL0lsqgSZzZiaLMTAClVb/VgGn70iDYlbGVTQxDgdsOGyhMDAI2xbGhVGnwgcQ+MMwKKZDOwZdesz/vQB3pBYJsD2pyBdXGEngSMsI5483ezBr+0wm74ccbirEFoieuuBRlgIB2KDsqixe58SiJriuKqGIuPkguXfQMBTrFP/gWfPqPsvOlKyJewRkVWYNYQ/wV0VBn3NIxhzO8AxGmSvwNWGxBXe5VIgQVFTcSTA3xR2gLjPh22qfWc4ZusWOarAvj1ZjE0EMIhTL7kASS3+RWSSbmlVBOi182aFBDBQFFRg2T5oBaPR/y8TPnaHA2Erpu+pRqcOM0ZjamO6IGN1BJNvzJOYpm5sBynDA3ZIEz7dyzlGKcKs4I4vNLeoA7cCavka92rNIjdfqvMFeq30o55tl/jK91zP+zf1ctU/jigRhtAfNrkMLGiEeCRZP4EPbQdNKonaJEWOr0sP8GGPWrf3Fzms8H85ua3uyT8MAsnV0l+ensXR9Obm4uI2dRcMz99LTy/FJDiL2bzOi5/lskLhfUNnakZewnVNYv8+FVgovNz+d7u6VlOd3nI4nTJQIrfJbubqpzVyQX7M6UmWVO+fenGztilip7Gv4L5mjxkv5BLE8CO4n6ck9WbkRYNJ+wClg5S3zJaJbfyjJqsuyG0d4Nd2zM17xbLbQjLmcEO6dvbNLoFtSM/UJ/isPU7OXJyBUdZRUv5RrWDgeiwBpoFnmzbFFZ3YPIVEK19lfx21Z0DzGEvclCSuwh6JfGybF96YW41btaSL8v2DUMM84xKEGaTUTz54o3jLEPJY+ElTali3rQZ5DJQ2R4ZSjZq7ctJNEiKQIfYtC2eCuALjSmaY7gzQ9JbC7jprjMDEYks7cC0kWXbT3W4Fls/ta+SqLeRoAWGFW22NqmNnDm7AbuSYOUyGHMgdkMVllFN3UFWbcpWCWr+SfwepfFwTlsuNRu0Ncf9Vgp7vywa6GCm7K1YrweTudCEH4eTeIjuApicIj4pgG/GYd6/g819Ms1UJEqBtCJSP2lfue2rUaKzdhphuGUUUeiOxtrkqKsZU8XRK0Ed43Y8ztX+knfJgXQMGBJLylHoaYy6WO6mKT7B1MK+ar671xHfQOMWYHvze4YIH5PPYJSEg8wzsCG3Ca/0ihXe0CjYADRSq3wOK0gO6zZCtjSGGw/Q4wM7ksINDXxAQ1ihrSstfkdXiFI2l0QrUQBQUzLXzKUEg4ugNoia4mnRkA/lxAbszGLvLwAAgjyQsY2IGf7ZVXBjxsLwLsaJB/GQuDrXwUlUzgWKBlWUMv4RuBRpA7yEX46N5KYlyJjOQoP3GqsQSfIvR4dhfQ1hAy0cpF80kRQJCtAqzyRDKaevzXq17ZZifYhfyaCxxLkOuyyKBgo0h0RsBErvl3b8DphzHT/RIdOK/it4SAlgPX/UHyN6bZCwCz9KbburzKAW8WAq3EYpaaeqNWhyia3VXIZGQpmtXcWG7e2gxnI8Zb4DQFUqc1sLcpU74SJcCP6A7dQOGB1A/rYtpYc8jwG9W0jDuYfe5QfKbcj8u114irYJTR7YJjTXC7cTzlTAXhbamsEvmtqmcTFCX9M48360LYOtwn4+C0eiUw4DmH1xK1MQIP9dI0VQWwFVDJQ1S11neerhyBpd9WkVisYMAo0PC9NBcn2b6anoXiBWoqnRIYyIkZau/ukomZPyIVgdTG7y5Bicl4C4ZmT3O6S5mteljHgyIZVuANBMwUxlSTtD/fez+7J5YOxTXHNB7X92Scb4mD6xVrUM7S7MKTRtj5WEgSGylafCluk1Q0thX4j70QGbGoCGi5VC3Pnomlce4OXnCYYnvBUlqRRbFQiHlA3VT8bTUZRrJ4y2y4spEuegKX5x55GMcIc5fmJkZMcXl+EQTaA8DdlwLRLAtFC+RyNXpkXoAJf8JBimyddoEnyNp56yfzjilYcpqJGAt6CYopDkpCqbwrCTjHXL7WEerQ+BLfWXuinuQKK0oilsXl0jzJQ2VnOftsOZKzqkAOQuDIc/yFL5bKk964OkAFIQHxmo1HnTLfcHCKlPEyY2ZjgVc0IeFdixWvhbEqscJcny8BZzLh0i+RhYJWp69nPyQWIQMJBfuk53XPf8poZy9aM6D9M8BmVgDCmN/mUWY24e7gWjB9yOM/CF26/qtOzRsNtRxFgmUPl/xFNUUBZLUtxYPihjjspyHI17mOQ2Uc182EZKwdCudJSz51Z+6GPX+ARYZDoK+5GHIVX4hwzeDXdgOpoRpuSTJbcdft7w6VFlYyBZMronTeOpKd4Q/JV851V3ozU79kZ+DvjjerOLnKqXGJlNkuAanNJOqm8Znc7atbGDKv37aZKhqIEOZhPccqF2pyRQJh5wgEPFRRXB1MRIVgpHI0UGu61BnAqh0DDdV8bLeh+Wy41pVF5DDdeK4rcVrqtlZhR/VEo44MeGS5PMX+mKrRDGjvfK8NwiRejt5hvmTDcHCZ+atjTjoizo+i4dG7vUqXy/uc3dlnKZ8LFtkzdM763eWlS7OtzdimGyOAAcW6EKPChhVLSdJyCV5BnKIuHcoRGoQilU5PNFQ6anRwO0WZPRbDzJOIfJaU4pYmEvGvGf5pY4U8SXO8ubtYXpDLWBA1dlsx4M2MXflw1IZWHw1XlPqun15s7uUgtH+ewyZ4MWZ0I6bBF82dLRVEyjPmxxKcz9VE0zD0ww33rRMIjTJksam9CDxpJEL2eO6CN+0ljUfoXyPsXcSGU4FwTEAHw+g2FlHubMPMoUma49wLocJiPxxjBYbKqgazfNgy/RXPnmZdp5kmYdoH6TuaCNDjKfRBKI4UYp2wgYw0WnML9rd+/db5p4Gc90PQZpJP2sxe4TebJwp7+MxgZqkjF0CK7ELKa0VW9Q1oY9UFOzXBptIP192N2Q+pJtSPM9v2GUCsslBPYNyqqEj6Wu6AAC0qKQ0hsaAVdNUGPb1DR3WPS6wWpaMfbz2CuPhZTTuEUknPTe95kRuBPrh4OBubLUJsyUqs4kX7u/tYjrcrQbPse91x2m5W01QHBN514ZMHxGqEvZgTtw57UjXbr+OJzK3D/xyDLBmojHhhEbLPQa7SJlTg+yLhlEwDhOErCRBcaHIjrC0o1sjhWNYA5pWdCxHM/I+eHPK/J9mA8sBkdBTpCIEWqSZq+x4+7zvKFcWKRlBOfBFXxRsr0QYWYeRBqRaJctb0rl5zRNybBgbeNP3NS6Qpif+7TXmTv2qpar5TXg7kdkU1rbNrfW7Uyq6cCDk2edMnW4CUhBL0p2qMkxfjiFsQy8Kh3H61o/zhPP5JOGH2bBNMniR6+xwoEAexEwUzNVDHuhTay3EjStZiVX+ySGef/RtFQqU3lhW4G2LYW247sUDWKTNoVPGZjC/ZCzzLbIUeu+uy6674IqxVh/iGsHSDaZe8tjAoV7FjUWHbAzDBYNTB4RU1PrYuDnCdmGZlxctak2NDm/VukxNjt1DrAJRn0ezgZxLsNwTTAU1ZikkESLOigmwbTHaXoco7Vo6VgETQOwGoEN2ja5q9qThCD2qhE8TaMxOY8CuZjwm00T5Cgexzl7kuDVzdppFBqzwgFSPnZGpyk38jSKNrIow8RicXKUkRVcHNJj74GPFhP2sbDdbTIrRxCpyeHWLtz3SPVKxvyOoPdF1JvFo4EV9aZUVHGbhtM7mWwcDgTvmoR08BfKCI8O8geDaBhPYvKVxCijp+gTMo/k+uKz/LyDmbef6eCvCO+TeJBJ0wwRxu6Q+Uk+nwIx5Ee/8IFIzPuKBmZcHXrDb8hgGEeDGMj88473EaJ/Amvg2/+RINCXeXF8fnF29Onw5O0JrJa//SjgRVI/LNvbUdID6wMdsx4Hc4BohkhKMjDY7+M0mYAUO//b1fuz0/cHl+/Rp8y2iw4lFLBZlOHpQV/DLVTOZOrXv8yHxcs8cBroVn8S//e///u//Q9gg96//9v/Eufz/A74akOczsbnc/hvkcyPXJGI3/A8mF90Yqb/4D90XszHw40Z9csoBBX4ySz4Y/oP8IiTII5ay8KUly7o5l+JzhzcwDWcJ20nUlP8y8KvOHBQBPbDEUxK2BtFQTINBrBdTcewuLKxYXpomaHf5nEfDRU+wpuB7Ds9ePsBZ7Fh5vfrkaG9E7VfAG82Ce/DeIRomRaG+vAiCjOkFyZWGQMGTt/3i9XcC/tferD8m0qkvD1tshAdJ19gfYNAsw9njosTppWHMtVpTM7D+fQrprwdXJ2cnYKoP3138is7Ne6SNP6K+nUUDEcxiXnex4IZlLB8D4Yh5lyRBvA395caJTdrXxPgH/OL1ibZNdAHsFruvloOrJeiiJtEbBKRRl7YK3mAdQ8m6LlLo2ma9COyEGArikLKWM2PeMyLzkP6F1HWD0fYquVvwprZ2t31AUc0mDoY65MAU2qGM6qguZ48Ce80SccA7ysRyxtH4aRzvenv7O828eAYank8P7bXXcGag3/uwzTGY60IY2vrzatXW/j91taO/msX/loRWPgYZ531VnlwBbVwfI/OmSfSfQHM1yQDa5zPYHgcgFaaSlqxE0qVLchcJViN7Q/REWaCxc4J/vS84rhe8dd2Q+Ochg9BetszV1Y4u0VhQM5zBnUZ/csMM37CkXdtUUbNOcm1d8DhnsnzlFQHKHYIz2b9lxdyLXi0Alb75g9YCR7z/0rtD+USWfJNV1JlKBO9AzQVAkkOlR1mZCCUWRfXQc2a0VQtWiuhpOkMm6SRDCBmvlFzBPrpj2YD3BtN2XIuWEX+dzWOdWMdiD6auNOoU8so0yQZsac/vLeiXQp9n4iF8tk6lUlb4ijMMf8WXqjW3mNTVzpRh+pA8f4WRVMQymDV3aMZliG9URVkspABbF57SX5HgYXI0LmG5DlKkynYT555vNScaE9hUwxBmlsGDCS6Z5xfbWIaQ3xPc9LB/cQwH4ePFH4mz/fN2hDMwHx7S9nWS8itxARsxsDeDrhzQ0IYwo/54SOJBmaypkK2YFKDQ4IrJU5sDm1qupdNNPWGIpmRV6hI+m0nu9Aj9kbRny3z/MWKzEDHOFxAWysBulJFWky3NE0ZMYb6hCex7NSg5xZAjx9RFBLwSPMMvTOcgne/G9Dp2WfPkfGkomcSVjhAAb3CcJuKnd5inpOlwRqNqjA67tddJGkTRM/kEl866NWCzLCRTKTkwN0MjfWVhPkAhtlswuZRRVC7CsVlCFXRDsC8iFQrjumgkKQaisIrExrd5YkAJR8JrwJ3YaLRmc3GHuw2QLgPvHufxGeDaHFv0UF3HMgJbDiGAAaFwMiU64/sACCHeVCBnvmynZeAAT2Ov0Zph8WD/p35B4Nw/FePaoLQ8STYjXZG6TcpBfRUEr6wA+iH8455Kr/WAzdKskxihX+ChL6chmkWHQIit2AQgK13iBWjQAOm5L6uAyQLmnSwfoQsPAL7QZRJ0TQLYFsQ6H10Byyvv8fodiHymC6lF50v/47j4M8cAkcMhjTDHPP6T3S8v/XTj/drCqCbJOBCaoaxa/TIfIPrJpoM/D5ycSB9WQYtbvsYssV0R+OhXDSGJbWCdY1+IOMAmj6MYCXme88nn9K6LQ4MSCdlY5VMfpUjQcEvBcDM56SXqhaKpYRllslToYmbhjJdrBKw03klqjsJWufO8jBltpHt2Uc89Ihlfnc597QidX1ZtugKcTpCUPe8YXjiqpKha3NWVzB8HOlL7CtZx3Mzg6+xg25pXdCX9ioYh1+iAMtxKce4N8RIgCOL0BSl5240z4etC9iboJXSBnrdqdqUsRAoTmF+LQMX3aIxG6yguAD77S3r2Cv8j4LnLCQ5k8CdkwfMkRv6ccL8XZGobEZ0uSWDDeihByCaFD+bAKadbTxoh0esg3AC72nPY4pwC6Avz2gAAa5Paar439vd+t5lVCkDBckRoKa4Lu9xQLWAyrlLBrBEezFsdED4uNaPSi0a+n3cQEpg8FOa/g1JslIu0tDHufalf91HR1LA5aSCbERlPYiCmfweDAQKydpxPiJhE8uLYf5UCBJwFGCNq6yj4B98ujq7+nR63AAZCrzvFXV5GmCkRMMIH5Yam74y4xRaIRJwAbGTPbzFABEXBpFxflkLy8jLncIyQpJbJ0mL16VEJK49Ib+SEtVI6eXFx7W41qoOO+qmVQn4dVljpY9lCEqJXR5vp0JnVWQbm2kd6kPWZfQrY9EhNZDbcVNU2C1ZiGlCAGaC2oOUd1lumFH9RqOhqjmo9KepD8zRBysMhIU1LI+hl7bfvpQltdaiM8pn4YgwT0adVrS+23jmHDMHgnIzrgK74/A2jaKMjR8VHjI4iYWrXKNIdXuYTRdfarfUxvilbFwo93JxHGKDpxjWIJ58wygZ4/CTcs73fPEOyPCVC0+q85pGPUfp7nNLi7LTXTcL5JFDM3DqHAVs26e2jbBpqaE+MGjGKmVj5/Bau/asWsW3xoEDOhBZDUSflXwGhDqTUQ3FOfNQAepl0eJVwtAvjSgvO9bTLltS9qd01q3NxzOaouJ8Tds9OmF+z4f16NidkfbQ1EiRF1SWzmiLIhHOOOSBblPTHVaMX2UTtDGlAWGqGscBRWA41H12FVx8OtWx6iqGVrmtZsKA08Q8uQuisQpARYI+tDT2AJUfLbfu0dJ1PqsybmvqORjLm9atzDb6RQwSPm4RP6pTpJl9zjsriqWYno1i/1s1lqaLqnN+3sFmicxbLe5//CgrWtfX6MbIPh0HV3xlHgsH+3zO/mBfHIxG8tw4ZWdhrYQQDTg8KI7Hi2TwUVZ5nkYyN+sXWMpYETrjksczWdi3VMhaV9qWWP7E8/GvfSEpHcF8ROlcxSgNrYgVw+WZIy6Qm2NZ2XSAqnaMAVOGToM+vzh7Gxyeffj08ZQqccLuCakTqIw1YSbtyfXdtTcr7BHhOq2eKtfKfYAlKp101h4+mw1hYqjiZfEeE0qHRnuun0JBJ+PEkZUYpTrjLCEtfWSytnrQUIlH1tKSFelot+5Wm4WdO2G44DQkI2svvgWDBTGfjnI/m/WwvG/mwWPcOXS8/aZ4DSbNMOyDIT5KUljsD3dxbnkTw0fYbmR3yYNGvw92PLR8C1tYKu93P44nnU38b/jYaVVRBV54LazigAB8/NVoWD3gFugxj/tfMthJgVQwoxSAoCaVCix3dqC/uxBjfLh3dfBFaPNnodnJjeTmLbcuu5/6qzSUiJAk7QPaTZy2Jx7/ddrsd9v+1nBRzUKU1QcyymjdWJIdSFToY/SNis3e2z+HCcgWnOs3LoW8R9qXQetzrjaPLN0Uc/WUwhJYvDHORxSReWIW9I3DYgvgR/W4nDIa3/o5+VpH4RxjV2CK4jMU2/BfbzWOnuIptKYYTONOa38TQCAn8w4AgDSqljaG6HucYuEVlfR1kMmYKrLcBmhZUi0j3fRaliugVxV+BtobG2eFk1SWI3Rh0PMKAL0kGTWsWjiZ87Up6J7rH2Q7WQx27yjxA7Z44sFjBRKxeVzJTn2mMMLgloKbsOkawQ55iofZYNXBOm61nMTgYkm0Np2VMA4zNNE9g9T/1GHY13G3If7BevWP6s1fWlSPMYYnb3hZmK06omV7MSby1DX25mMIwwlFOWm6vZhSD2Oq+kwps3TqBP15+Ethhw9n8JH5EBBbegylcOy3BU2TJ7ngGnHr+pjkAdjRwueBAX0n4WQpUPwoMDmygK2ePQN+YYS240nmpuwWqbqqhZH9yqsU2LlYVwE1qUt6Je6hNcVY4oSk10TqLkDCjEuDVRvilQh7GbUwwiJiXeCT0tC7jeXneYgXkZnxSIjs1OCGXhpTuj5jBsQhisF/EUmPV+I6Ei2a2wL+mlYZcOyrV2KryfkxLVONWam+VTST4R8j33cNqBS0NomaxGP9ysRtoALs+WJeyzQAbEx/aA52iVoNB6/+4J3IURzeThJOSIMtzS94KUt+B2Yj2oRkfheYg9gGG3FtUTIsSD5YpgVIhy2qTMYGRgt+7/i7BpVmGeXe4Iiv6V+SLf5JGIU7EfL1ZtdHmN41Ch1Yi+q/N2vr60g+ZbHcpuHcNgCMj7G7Si5qCvnKisTdrCXrVbBYX8bjDknABqpJ44fWpIdGD4YqPdBdVK9yrWIvolHMhj5eaHN43HkCjmj728NFCalW17/DQ8mGBJCCBoaBhO2QwIJZGk3BRPBfSz8sYSnV0vMg/3VVmCeTfj1Uy9qoodGJ3D0rg8NsiH4zWDq9GeeMlzsYRbco2aWBYRsdlj1Ssj3oyIaiucEhK9kdhjscVy6vyCqLBO9mmKbPGCM/Qot/ryVRtgJWXunww1znZBegYmjq7VgEHWPqa+RVG83AYSFsEMFYkKToMISGig9ZQ5UeDf6IdT6V7NvklH5+XplmYdsC9rEZdAVx3Ul1/NTsoMHJrXTvBn3ynOqWNoF7UYnxLYc381nGYvnT6dHxu5NTTDFelBUdpsLGEzddaApaIMd/BVizUl0DIhFvMk9ct6We7rpllCVK6nYRhlJ9Z8lqIPWdJFrLWjeV1AJxoEz15zX3vKwI6Lume5lRZvCBcQ1LzWyHU3emz34rTbGluYppZRGJOzDecz3p7kqqwRSKBEVNqjHPZXDTahXzvQrYEvtkHuItZzGd6E2p4JCz3Sy/Vrrg4uxQnP1+8axyuaDhWoDPi2ko4J1fuOCofuwj1f14dM8OwoZZahi9nd4HjQCPb9N44EltuPUNW1/momkaJPfpyiqnbLlXGOsFXG131pnrRUIR8lRAPn95WJYXCYXdm7KOmIxDldSXKgDYqc0G0E1RwnAIVOopTzb0c/jUQ5MK1lAPBt7ZtFJKUk7cpuK+sC0bsk/SuivA9pXS7jXMwhRY1Kvs0uzKTKELp1ldX7A/kLjo9vOA/KvPZyjETokQC11Ot0Mvu0dnfVM+iGnuQtAyl5HQOKPTYZFnwWhghLS6zojjlz/XNoJ0zNFVcg9pginKhAj1NVnnbji9LHPSZdyorDMekKJyo0SOAxU73XkhfvakyngBAMdLITH918BpjnJ3wKWpDUzC9BbdjhIZw7OrrkoEo8c+BfvIaY+PFae/u4b1Rx7rTvXFYR6zRVMipTIhME0lrHBMrprQKAvOUM3JwovMCzJAksl6EnhjJPy+J+lnriTGLiD+KnkC6F3Dv7KSorQw4OxmvqJRVWxx5tIA7o+S/nXFxVpNvEJragpmhKlE983asLVOsggWUJfXMJhKlehXuSmqpsL1jPfHolO6te1l82UCC9CJ+90QjfBDhy1/E2UrbtEfFzELx/PitCPUisZ20SyjwoaSXKVCCs8kZxlbDZeXnizG+MGxeWlaY52hHPads1Q5NozrbImZ9L0GKP+5jT46p04WNPFa0HWZl+j2oU5ry7uNGESpclGgahq5ZXfrY+FGjJ0C+YbKVVUJ21Lrqow5Joeao8pSRPql0w/tJPWRfyXtruNu4cNlbu26X05VgODZz7Flt7JjYwvbFmpVmKCdBrRcHEDaxdDWCrejV5btsm878t8Q/gXUxXdtV809thuG7LoKqNieVC2cCilmvl3mZzXbobT917oQRLeikwhVLYrm6FlnblN7TlcLsjjiaIkHxEgZp7NDLPDxLIWWnktUs1dxF2P99YsNY686xDGqi4Zl/ED6gRUm/yA8ZU90nCIZxh2EDdMLrC4tNbOSfpr0W1F0aAextCgdKJQ0INcf3x8sm2vrqGFslKNBUCxoznepFAa6eUkelEMk9jWsjtJs1CBs7a8ZjushqNG+GrVvPUMi1N2zHYkM8plrbtWgrewchbN5qetPQfjbUR22CjTVfbY/FMVvQa10d2+BY821vs/yE57qKaCsY8AzuUVZ0MdTv7YcLyw3XCEMGf2CSnXizZ6vMSzQULE4tzdL7hQLznpMdoQ8SlnklaGEkk8rHVQubPai6Bm05N2GWA6pIjK1NA5VbTLwsWJQzANQ7XTOFw0xPpo7jacR5qQTeN5vu1B04mcw5vNRbARRJniLTKvW5uameKX36xtLxNxq6YbGwQ3DUFge5HMDe/J3+eQEPy+cLrjwDYVhlGx3vS+k6jx359UslU1qNM205LI33y0g365KMjJzDysOBxXIOZF+WZHZNDCKFDxVJXJgJi8ajny32nLZcY+0H9gHmVYpx1x2o8d8dm6gbg9xsgetKw7M7NqqNMOaosDVRZAPFQ3EU0HEhXmch3ww5r0aZr3zirOEzrmuWqZsfAON0Mmy1CRtLJut76KCUWLU6LWKDkWplBk112xW3Ho00d5L+/PyOVk5cJnRpDJo+caeyVzSoijW96JR6wLYReKpLoCN6TWcftxW+VSVlY8lAi+4ElizS+3ZjzJdbPb6Bqdw+bgl9F0cVzDfvPTogisPWBazGOCMiprF3H5+Ja9kuXyXHlkKdxVjfkUM1RoIwjyY5RSiwoOu5MueJA8e/vEVj3HDS1zEyRD9TkDjRX3srKTUYKFcQ4PiAgWzni5+2a24SKHW21tcTGMdA3FrC5PVWGjN64JW3eU5TTUfqU2xfZMfqT1TOFVehVGxWXbvVNsuzq4Ieyg/Kd963xcHaijoIYeVEcuSemBS420s0xCPKFZYGnI37VgYVjledXVoDTGdTTB7qNEAMfd8zaoNXLO0PWpau5B6xq/cEDS1Ud+sMYqXAbRM32qrt7mCcdo1ogw4D3OXQbEoNlPKjEfjrsHcHRLNuapr9Y6iTpzQQbYgG9QBy/KBNxgkQ8xiWwkiVwjKlHeAVnZDuyaZBkpaUeFu7s6P0S+12VW5eItn+zJz+E1W6jZckq66fCn/Cwa83JUG+POBi5fYzDoEZNaTX9F+Fhur0H1JQEOOBAvCmwEel4ilqsXaelJDNjIkVnQqvbTcbDnVQg4EuaOUjNFtrAbScgJVADTed1eDWPhnKsCplyvCYgdKBRwjwLWoZkBLUugXFW2XLAHdJrAWQ29Oy3S1teDfpsls2pt7esYb188F78jbDPh410qccfILXX9VebpldeTtlbz6UZc3biIqqGoplKUsuVb/LaRyl93QNfqrRlVVq6Ruw0xvBYsXneGx219TmA9D4loxj9K0Uzw3BHsX79eY0gB3S+c7ihRVf3O3YWTEXDLnmUc7DAq3BXYs/s//FvLA9uWRwMFlGR9JU/cNrpzsYl58XCWZyYyQ84qSMhusnAkDOp0kr2bWaXyf5PI2Ac2wxQkrJcVkrRx8gFOGm2P26pePBr2UrVSVhuLYFCL53KGp5YehCm0L08h/fM9xJ/f8z6WkSQ07AHXXWSq9a7kZU99yasoYTVkh0bxg1SAgGhsOsK3rd1c4XEVfmseqjAwZroykTuidbhwsMwCXHaiyuZ4S4HAly6oZMJPhY5ET7ZDru5ZJIQ2HLTRlc2CnlZcJmQLFKTO082WhRXIWOJeGffthx+fyhU2rUa+grddNsWstoSIdr6kNL7pb7hFzzxlAmXNkDtNPs8MIq2fPXtoGWVW+hMHBP/lcpdHTi89X7taer3SgvkTwOJ+SvPHkxFSUMyof61nh1OWLj2n+Bx/XfOGxzf2Gu7rqjje4RxyWy5OCjRX/KteGI1TslIeas5dlt4c26ESOVQLZAyFR+o/xhLzxxQUZ3sbt8cbV69kvqtqPeQhf1qIlkfPu04cPdJ21UTu7F2bR3o755Oso7hWEQMFDOXVDLBF2A3g8WVF+owS1pNU6leyXif2yxMRQPOmQ/8IXp4muRtMU72KsMXCFoyluzVhHvAZG7TLqoXxDKVVsEJ+fqguPLD774si5U/bJqaaxkKDlzULq2uPPT2ZVkMXnX3QBhG2jjZ0HBd2ZhBDbVDhDXsrRLFct8MWJLKhhEiHOhCyqATZDKis0YaGF/CGR5RCULxvrIRSBFTZp5IbTF5+dJMXPagQE8F0LO5pN6EYGIEsPTH8EB0TAgA0GrRlXBY++ulDZdXmi75zwsKIWVvNTgVg2GLFAaRqNE4xwUSkEErt0LYnZrrgNtzfPkYdxKeF9szI3TRYWyu/ALp6mAG2C+xC/4LpzMOZoYuXCAW4gtfhqw/T7bXw2bk5h6jV1WQa54ousP+qS9gZWqLwpKL0oa8oNcNYs0iPF4eXvG+env7KcNI8aHh8eb7ylTCMkw8XZ4cb5BdJr/T5bT7EeY61/03Bs8g284rOWQjAgbPJZSqAN4HTY1sCULtnjNGX1Wrnagfrr1JqjTPpsGlV5AQPTF0bCMtYuhWbQAbkDxVNRtkze56eC4ZQ3L9QNHXzHMQ6Xv3sI0/H6bCrL+d8lsxHfE92L1MzzdEjkC+4eRPdYg01wNLFY1MCQgPUcAEz6dxg2MXjjM0o8HywtINZgkHHtELNcSYXARLzc8lW/MLSD04MPf7s8uWSImp80M+BZ1F6SfLFUA/ISLhOOV9FhnFiVxL4qKrQpjNjFCOPGJLcEdvOwhqiX2QQg0CW6coD8f/indRPPxfHB0cdjfzxwLgRlQV5fArX6Xp9CZfFJn8qrfX5whSqlGmRa6vGny+OjAEACeaQ/Jq6rQVTO5MabQM8+nn84vjq2ixz9gFpW3x2ZW6i6dBQ1HXBgwZjA5rIZqKuh1Ky7jqlkwT17+1PFF4UAMgldE02qbW77vIrL6Wq80i8M67ATW9L0L1xTx9gqbZh240ZVnZ16W9cr58ZUZrYu93EvsaUr9mOwJTXAVr1lC/dlUAurfwlwo9EqfSwpDbDk/NZSmEvybdTdpTKtAuaZps8ME2guMJI/pGzj26XN+yldPEqNcQF6+J8AdzKYQStP+FfLTlp8gUIhwGwJvPDlW2WoBmRltKqnlmCSJKFSePyne1aTi4Ppe+ElCaUP5N3ByQd9lBMTmfj10uqQdNZfEtuId+tBayDiSf5lX3kEMvz47dnZb4FUCX98OHkbvN3bIbeN1HNKvxKF7XJy6lVFS5lcUpFOolvhHZ2YpG3C5wKnfDkIXtmga1STiAs2tzb9eDqf9OzbmEn9WxlMuJ2iyrpjEBNZ5vGGy+/t7cgqvnUjV44P2vvDZtK4cBZ7wVR4eJpZV9/CZOFD+Y62n/pe0j4fFrG4XDZWFQIx4+AaMxj1Xe9/fmZe/vyi/N3hcnBPdW8X2JFbIafigpShIOSNKyKexV/6UCltczl0uj4BmuE8uHTrlvmppk69MYNVNep1BftWQ/wFMbwhSbG0bn2N9AEbaZ6BfNM6vUrsqGvirZNA9y1leiyVSWVjaol995+g8uhPqPy5rPymW9jzJcVBF/a94/YNvlRNk4QB3b0rJ91P8a4psE9eYfajJRKoEGah6XhrCY9ISxYF6t+dfDgOPh6cnrw7vrwqjILD98eHv11++njpK7QXbkEsyWBFxF2d3JIlOIu7YiWyL7gmFkgJyjYg74AGaWti+2ZXkxPYCyQzmezChTK7uOpO1EpClILK1idlIlmbL4NlaEX7f4cpR52J91Vf/5m/+XN3IQQ/QLzhp7RKZQZaccnsqpIhHGEFMKzZzrKAr7krptawzvR+Q7lL4a/8MZd/mRcFKLVX5Hbz8d2m8jVyGmbGCd5qYyAvkcYh2/7DwNxVYw/yro5V9MoK0JUHACGT5jfr00To8eSQOY/BR6URsP8YlhJ+6OfjqXVoFX1D8Bz5y/8jnmLVck9DQgI9cICW9T2GHnTrk/Pg6Pjdh4Or4yO6EERK33Jw8WVL3NB/FUvdc6ZFgvd5lPrOB6wT4bBLTWxBYs38LS8hgGd0z5VH159Yc9BYNRZl/1MrOkzZUSqeY4+1XRl5WZL6VD3E6mskvnnQhSY06wHXXVvSQ8kqL7Tw73ZRsNSNv4x5RR37b0C6FuH6KwBegqSh8380eoZBIq9PVYcHVkfPqL/8Y5EzjJmiWLfhyKlGcrn8qRcrsDg8NTB0nWH8uYH+XDQA8H6Oih2wChE7QnVDGbobT7bJq1wp0qZQ3eHjEdZhW/GOMOjxgD/FKID2oYhhCGMdwA5SD9eKruqnevfCQr18Cy1doIFXT2tKYbKI1GI4XqkNnA3/hth69Wprs+23hgvxMX77XdW1j6xb/lRJcfTSZtZN206wS/qW7y2ntS9Ocj6WzNECisC9Lwp2C+uUMsah1lWwSMYbmoJyGorIYqZv1EMNuqGUNIYPMyuAh/rEjl6ls4lvRodYC8my4RjIMJz6sjp6EzQdjhqsKZWTLWMDX/CmSX0zuh0B6i66Fonx2giMmEyjPv4axGDyh3Py88G0yPuht0EhTufcsgEIjcLJ7Sy8xRZTaoEbOf6Cf2+vLYpWQTwZJggcd9PjmE6bo+DDR/KrWIORm6q19jZAoJvFwBaEPR89A0NrTt4rGADzDVqKG4/r1Vjg7x5eARileBs9nUxNzbfTOd1YmgWj6JHexAp9A5G1bb8F/7+3tlggPD4ksdbeKX7gKfMEPt9d/H9DqrH2"
notebook_dir = out_dir / "notebook"
notebook_dir.mkdir(exist_ok=True)
notebook_copy = notebook_dir / "densenet121_transfer_5seed_020.ipynb"
source_book = json.loads(zlib.decompress(base64.b64decode(NOTEBOOK_SOURCE_ZLIB_B64)))
for cell in source_book["cells"]:
    if cell["cell_type"] == "code":
        cell["source"] = [line.replace('NOTEBOOK_SOURCE_ZLIB_B64 = ""',
                                       f'NOTEBOOK_SOURCE_ZLIB_B64 = "{NOTEBOOK_SOURCE_ZLIB_B64}"', 1)
                          if line.startswith('NOTEBOOK_SOURCE_ZLIB_B64 = ""') else line
                          for line in cell["source"]]
notebook_copy.write_text(json.dumps(source_book, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

save_json(out_dir / "analysis_manifest.json", {
    "archive_version": "v1", "experiment_id": EXPERIMENT_ID,
    "source_experiment_id": SOURCE_ID, "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "class_order": CLASSES, "seeds": SEEDS, "checkpoint_sha256": checkpoint_hashes,
})
manifest = []
for path in sorted(out_dir.rglob("*")):
    if path.is_file() and path.name not in {"FILE_MANIFEST.csv", "CHECKSUMS.sha256"}:
        manifest.append({"path": path.relative_to(out_dir).as_posix(),
                         "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
pd.DataFrame(manifest).to_csv(out_dir / "FILE_MANIFEST.csv", index=False)
(out_dir / "CHECKSUMS.sha256").write_text(
    "\n".join(f"{row['sha256']}  {row['path']}" for row in manifest) + "\n", encoding="utf-8")

allowed_analysis = {".csv", ".json", ".md", ".png", ".txt", ".sha256", ".ipynb"}
for target, include_models in ((results_root / f"{EXPERIMENT_ID}_ANALYSIS.zip", False),
                               (results_root / f"{EXPERIMENT_ID}_FULL.zip", True)):
    temporary = target.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(out_dir.rglob("*")):
            if path.is_file() and (include_models or path.suffix.lower() in allowed_analysis):
                archive.write(path, arcname=(Path(EXPERIMENT_ID) /
                                               path.relative_to(out_dir)).as_posix())
        if include_models:
            for seed in SEEDS:
                archive.write(checkpoint_paths[seed], arcname=(Path(EXPERIMENT_ID) /
                    "source_checkpoints" / f"seed_{seed}" / "best.weights.h5").as_posix())
            archive.write(source_deployment, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "deployment_model.keras").as_posix())
            archive.write(config_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "experiment_config.json").as_posix())
            archive.write(lock_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "validation_selection_lock.json").as_posix())
    with zipfile.ZipFile(temporary) as archive:
        if (archive.testzip() is not None or
                f"{EXPERIMENT_ID}/notebook/{notebook_copy.name}" not in archive.namelist()):
            raise RuntimeError(f"Archive verification failed: {temporary}")
    temporary.replace(target)
    print("Verified archive:", target, f"{target.stat().st_size / 2**20:.1f} MiB")


Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-020_ANALYSIS.zip 1.8 MiB
Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-020_FULL.zip 218.9 MiB


## Done

EXP-020 reads the EXP-018 Validation lock and five checkpoints. It writes
only Holdout (3) predictions, four-class summaries, plots, manifests and
FULL/ANALYSIS ZIPs. No training or Final Test is run. Holdout (2) and (3)
are different datasets, so their results should keep separate provenance.
